# Visible Points Depth Analysis

This notebook analyzes visible points and their depth characteristics:
1. Loads metadata from a sequence
2. Given a frame ID, extracts all visible points (excluding newly sampled keypoints at that frame)
3. Computes and visualizes analysis on:
   - How many visible points have valid depth values
   - Depth situation of pixels around visible points

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from typing import Optional, Tuple
from scipy.ndimage import uniform_filter

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from point2pose.io.sources.dataset.datareader import Ho3dReader, YcbineoatReader
from point2pose.data_types.frame import Frame

%matplotlib inline

## Helper Functions

In [ ]:
def unpack_ragged(name: str, store: dict, dim=-1):
    """Unpack ragged array from metadata storage."""
    try:
        data = store[f"{name}_data"]
        offsets = store[f"{name}_offsets"]
        lengths = store[f"{name}_lengths"]
        out = []
        for off, L in zip(offsets, lengths):
            flat_data = data[off : off + L]
            if dim == 3:
                reshaped_data = flat_data.reshape(-1, 3)
            elif dim == 2:
                reshaped_data = flat_data.reshape(-1, 2)
            else:
                reshaped_data = flat_data
            out.append(reshaped_data)
        return out
    except KeyError:
        return []

def load_metadata(path: str):
    """Load metadata from NPZ file."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} does not exist")
    data = np.load(path, allow_pickle=True)
    return data

def get_frame_visible_points(data: dict, frame_id: int, exclude_new_keypoints: bool = True):
    """
    Extract visible points for a given frame, optionally excluding newly sampled keypoints.
    
    Args:
        data: Loaded metadata dictionary
        frame_id: Frame ID to extract points for
        exclude_new_keypoints: If True, exclude points that were newly sampled at this frame
        
    Returns:
        dict with keys: 'points_2d', 'points_3d', 'track_indices', 'valid_depth', 'visible_mask'
    """
    # Find frame index
    frame_ids = data['frame_id']
    frame_idx = None
    for i, fid in enumerate(frame_ids):
        if fid == frame_id:
            frame_idx = i
            break
    
    if frame_idx is None:
        raise ValueError(f"Frame {frame_id} not found in metadata")
    
    # Load tracked 2D points
    track2d_list = unpack_ragged("track2d", data, dim=2)
    if frame_idx >= len(track2d_list):
        return {
            'points_2d': np.array([]).reshape(0, 2),
            'points_3d': np.array([]).reshape(0, 3),
            'track_indices': np.array([], dtype=int),
            'valid_depth': np.array([], dtype=bool),
            'visible_mask': np.array([], dtype=bool)
        }
    
    points_2d = track2d_list[frame_idx]
    
    # Load visibility mask
    visibles_list = unpack_ragged("visibles", data)
    if frame_idx < len(visibles_list):
        visible_mask = visibles_list[frame_idx].astype(bool)
    else:
        visible_mask = np.ones(len(points_2d), dtype=bool)
    
    # Load valid depth mask
    valid_depth_list = unpack_ragged("valid_depth", data)
    if frame_idx < len(valid_depth_list):
        valid_depth = valid_depth_list[frame_idx].astype(bool)
    else:
        valid_depth = np.ones(len(points_2d), dtype=bool)
    
    # Load 3D points
    track3d_list = unpack_ragged("track3d", data, dim=3)
    if frame_idx < len(track3d_list):
        points_3d = track3d_list[frame_idx]
    else:
        points_3d = np.zeros((len(points_2d), 3))
    
    # Get track indices (assuming they're sequential)
    track_indices = np.arange(len(points_2d))
    
    # Filter to only visible points
    visible_indices = np.where(visible_mask)[0]
    
    # Exclude newly sampled keypoints at this frame if requested
    if exclude_new_keypoints:
        obj_key_point_frames_list = unpack_ragged("obj_key_point_frames", data)
        if frame_idx < len(obj_key_point_frames_list) and len(obj_key_point_frames_list[frame_idx]) > 0:
            # Get frame IDs when each keypoint was sampled
            kp_frame_ids = obj_key_point_frames_list[frame_idx]
            # Points that were NOT newly sampled at this frame
            not_new_at_frame = kp_frame_ids != frame_id
            # Combine with visible mask
            visible_indices = visible_indices[not_new_at_frame[visible_indices]]
    
    return {
        'points_2d': points_2d[visible_indices],
        'points_3d': points_3d[visible_indices],
        'track_indices': track_indices[visible_indices],
        'valid_depth': valid_depth[visible_indices],
        'visible_mask': visible_mask[visible_indices]
    }

## Configuration

In [ ]:
# --- CONFIGURATION ---
FRAME_ID = 118  # Change this to analyze a different frame

# Path to metadata file
results_dir = '/home/justin/code/point-to-pose/results/ho3d_single'
video_name = 'MPM10'
meta_data_path = os.path.join(results_dir, video_name, 'meta_data', 'meta_data.npz')

# Dataset paths (for loading RGB and depth)
ho3d_root = '/home/justin/data/HO3D_V3/'  # Adjust to your HO3D root
video_dir = os.path.join(ho3d_root, 'evaluation', video_name)  # Adjust if needed

# Alternative: If using YCBInEOAT dataset
# ycbineoat_root = '/path/to/ycbineoat'
# video_dir = os.path.join(ycbineoat_root, video_name)

print(f"Loading metadata from: {meta_data_path}")
print(f"Frame ID to analyze: {FRAME_ID}")

## Load Metadata and Extract Visible Points

In [ ]:
# Load metadata
meta_data = load_metadata(meta_data_path)
num_frames = len(meta_data['frame_id'])
print(f"Loaded {num_frames} frames from metadata")

# Extract visible points (excluding newly sampled keypoints at this frame)
visible_points = get_frame_visible_points(meta_data, FRAME_ID, exclude_new_keypoints=True)

print(f"\nVisible points (excluding newly sampled at frame {FRAME_ID}):")
print(f"  Total visible points: {len(visible_points['points_2d'])}")
print(f"  Points with valid depth: {np.sum(visible_points['valid_depth'])}")
print(f"  Points without valid depth: {np.sum(~visible_points['valid_depth'])}")
print(f"  Valid depth percentage: {100 * np.mean(visible_points['valid_depth']):.2f}%")

## Load Frame Data (RGB and Depth)

In [ ]:
# Load frame data
reader = None
frame_data = None

# Try to create reader and load frame
if os.path.exists(video_dir):
    try:
        # Try HO3D first
        if os.path.exists(os.path.join(ho3d_root, 'models')):
            reader = Ho3dReader(video_dir, ho3d_root)
            print(f"Created Ho3dReader with {len(reader)} frames")
        else:
            # Try YCBInEOAT
            reader = YcbineoatReader(video_dir)
            print(f"Created YcbineoatReader with {len(reader)} frames")
        
        # Load frame
        if FRAME_ID < len(reader):
            if isinstance(reader, Ho3dReader):
                rgb = cv2.cvtColor(cv2.imread(reader.color_files[FRAME_ID]), cv2.COLOR_BGR2RGB)
                depth = reader.get_depth(FRAME_ID)
                mask = reader.get_mask(FRAME_ID)
                H, W = rgb.shape[:2]
                mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)
                intrinsics = reader.K
                depth_factor = 1.0
            else:  # YCBInEOAT
                rgb = reader.get_color(FRAME_ID)
                depth = reader.get_depth(FRAME_ID)
                mask = reader.get_mask(FRAME_ID)
                H, W = rgb.shape[:2]
                intrinsics = reader.K
                depth_factor = 1.0
            
            frame_data = {
                'rgb': rgb,
                'depth': depth,
                'mask': mask,
                'intrinsics': intrinsics,
                'depth_factor': depth_factor,
                'H': H,
                'W': W
            }
            print(f"Loaded frame {FRAME_ID}: RGB shape {rgb.shape}, Depth shape {depth.shape}")
        else:
            print(f"Warning: Frame {FRAME_ID} not available in reader (max: {len(reader)-1})")
    except Exception as e:
        print(f"Warning: Could not load frame data: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"Warning: Video directory not found: {video_dir}")
    print("Frame visualization will be skipped, but depth analysis can still be done with metadata.")

## Analyze Depth Around Visible Points

In [ ]:
def analyze_depth_around_points(
    points_2d: np.ndarray,
    depth_image: np.ndarray,
    window_size: int = 5,
    min_depth: float = 0.05,
    max_depth: float = 2.0
) -> dict:
    """
    Analyze depth situation around each point.
    
    Args:
        points_2d: (N, 2) pixel coordinates
        depth_image: (H, W) depth image
        window_size: Size of neighborhood window (odd integer)
        min_depth: Minimum valid depth
        max_depth: Maximum valid depth
        
    Returns:
        dict with analysis results
    """
    H, W = depth_image.shape
    half = window_size // 2
    N = len(points_2d)
    
    results = {
        'point_depths': np.full(N, np.nan),
        'point_valid': np.zeros(N, dtype=bool),
        'neighbor_valid_counts': np.zeros(N, dtype=int),
        'neighbor_mean_depths': np.full(N, np.nan),
        'neighbor_std_depths': np.full(N, np.nan),
        'neighbor_valid_ratios': np.zeros(N),
    }
    
    for i, (x, y) in enumerate(points_2d):
        x_int = int(np.round(x))
        y_int = int(np.round(y))
        
        # Check if point is in bounds
        if 0 <= x_int < W and 0 <= y_int < H:
            # Get depth at point
            depth_at_point = depth_image[y_int, x_int]
            if np.isfinite(depth_at_point) and min_depth <= depth_at_point <= max_depth:
                results['point_depths'][i] = depth_at_point
                results['point_valid'][i] = True
            
            # Analyze neighborhood
            x0 = max(0, x_int - half)
            x1 = min(W, x_int + half + 1)
            y0 = max(0, y_int - half)
            y1 = min(H, y_int + half + 1)
            
            neighborhood = depth_image[y0:y1, x0:x1]
            valid_mask = np.isfinite(neighborhood) & (neighborhood >= min_depth) & (neighborhood <= max_depth)
            
            results['neighbor_valid_counts'][i] = np.sum(valid_mask)
            results['neighbor_valid_ratios'][i] = np.sum(valid_mask) / neighborhood.size
            
            if np.any(valid_mask):
                valid_depths = neighborhood[valid_mask]
                results['neighbor_mean_depths'][i] = np.mean(valid_depths)
                results['neighbor_std_depths'][i] = np.std(valid_depths)
    
    return results

# Analyze depth around visible points
if frame_data is not None:
    depth_analysis = analyze_depth_around_points(
        visible_points['points_2d'],
        frame_data['depth'],
        window_size=5,
        min_depth=0.05,
        max_depth=2.0
    )
    
    print("\nDepth Analysis Results:")
    print(f"  Points with valid depth at location: {np.sum(depth_analysis['point_valid'])}")
    print(f"  Points with valid depth in neighborhood: {np.sum(depth_analysis['neighbor_valid_counts'] > 0)}")
    print(f"  Mean valid neighbors per point: {np.mean(depth_analysis['neighbor_valid_counts']):.2f}")
    print(f"  Mean neighbor valid ratio: {np.mean(depth_analysis['neighbor_valid_ratios']):.2f}")
    
    # Compare with metadata valid_depth
    if len(visible_points['valid_depth']) > 0:
        metadata_valid = visible_points['valid_depth']
        depth_image_valid = depth_analysis['point_valid']
        if len(metadata_valid) == len(depth_image_valid):
            agreement = np.sum(metadata_valid == depth_image_valid)
            print(f"  Agreement with metadata valid_depth: {agreement}/{len(metadata_valid)} ({100*agreement/len(metadata_valid):.2f}%)")
else:
    print("Cannot analyze depth around points: frame data not loaded")
    depth_analysis = None

## Visualizations

In [ ]:
# Create comprehensive visualization
if frame_data is not None and depth_analysis is not None:
    fig = plt.figure(figsize=(20, 12))
    
    # 1. RGB image with visible points
    ax1 = plt.subplot(2, 3, 1)
    ax1.imshow(frame_data['rgb'])
    points_2d = visible_points['points_2d']
    valid_mask = depth_analysis['point_valid']
    invalid_mask = ~valid_mask
    
    # Plot valid points in green, invalid in red
    if np.any(valid_mask):
        ax1.scatter(points_2d[valid_mask, 0], points_2d[valid_mask, 1], 
                   c='green', s=20, alpha=0.6, label=f'Valid depth ({np.sum(valid_mask)})')
    if np.any(invalid_mask):
        ax1.scatter(points_2d[invalid_mask, 0], points_2d[invalid_mask, 1], 
                   c='red', s=20, alpha=0.6, label=f'Invalid depth ({np.sum(invalid_mask)})')
    ax1.set_title(f'Visible Points on RGB (Frame {FRAME_ID})')
    ax1.legend()
    ax1.axis('off')
    
    # 2. Depth image with visible points
    ax2 = plt.subplot(2, 3, 2)
    depth_vis = frame_data['depth'].copy()
    depth_vis[depth_vis > 2.0] = 2.0  # Clamp for visualization
    depth_vis[depth_vis < 0.05] = 0.05
    ax2.imshow(depth_vis, cmap='jet', vmin=0.05, vmax=2.0)
    if np.any(valid_mask):
        ax2.scatter(points_2d[valid_mask, 0], points_2d[valid_mask, 1], 
                   c='green', s=20, alpha=0.8, edgecolors='white', linewidths=0.5)
    if np.any(invalid_mask):
        ax2.scatter(points_2d[invalid_mask, 0], points_2d[invalid_mask, 1], 
                   c='red', s=20, alpha=0.8, edgecolors='white', linewidths=0.5)
    ax2.set_title('Depth Map with Visible Points')
    ax2.axis('off')
    plt.colorbar(ax2.images[0], ax=ax2, label='Depth (m)')
    
    # 3. Valid depth statistics
    ax3 = plt.subplot(2, 3, 3)
    categories = ['Valid\nDepth', 'Invalid\nDepth']
    counts = [np.sum(valid_mask), np.sum(invalid_mask)]
    colors = ['green', 'red']
    bars = ax3.bar(categories, counts, color=colors, alpha=0.7)
    ax3.set_ylabel('Number of Points')
    ax3.set_title('Depth Validity Distribution')
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{count}\\n({100*count/len(valid_mask):.1f}%)',
                ha='center', va='bottom')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Neighbor valid ratio distribution
    ax4 = plt.subplot(2, 3, 4)
    neighbor_valid_ratios = depth_analysis['neighbor_valid_ratios']
    ax4.hist(neighbor_valid_ratios, bins=20, alpha=0.7, color='blue', edgecolor='black')
    ax4.axvline(np.mean(neighbor_valid_ratios), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(neighbor_valid_ratios):.2%}')
    ax4.set_xlabel('Neighbor Valid Ratio')
    ax4.set_ylabel('Number of Points')
    ax4.set_title('Distribution of Valid Depth Ratio in Neighborhood')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Depth values at points
    ax5 = plt.subplot(2, 3, 5)
    valid_depths = depth_analysis['point_depths'][valid_mask]
    if len(valid_depths) > 0:
        ax5.hist(valid_depths, bins=30, alpha=0.7, color='green', edgecolor='black')
        ax5.axvline(np.mean(valid_depths), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {np.mean(valid_depths):.3f}m')
        ax5.set_xlabel('Depth (m)')
        ax5.set_ylabel('Number of Points')
        ax5.set_title('Depth Distribution at Valid Points')
        ax5.legend()
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, 'No valid depths', ha='center', va='center', transform=ax5.transAxes)
        ax5.set_title('Depth Distribution at Valid Points')
    
    # 6. Neighbor depth statistics
    ax6 = plt.subplot(2, 3, 6)
    neighbor_means = depth_analysis['neighbor_mean_depths']
    neighbor_stds = depth_analysis['neighbor_std_depths']
    valid_neighbor_mask = np.isfinite(neighbor_means) & np.isfinite(neighbor_stds)
    
    if np.any(valid_neighbor_mask):
        ax6.scatter(neighbor_means[valid_neighbor_mask], neighbor_stds[valid_neighbor_mask],
                   alpha=0.5, s=20, c='blue')
        ax6.set_xlabel('Mean Depth in Neighborhood (m)')
        ax6.set_ylabel('Std Depth in Neighborhood (m)')
        ax6.set_title('Depth Consistency in Neighborhood')
        ax6.grid(True, alpha=0.3)
    else:
        ax6.text(0.5, 0.5, 'No valid neighbor statistics', ha='center', va='center', transform=ax6.transAxes)
        ax6.set_title('Depth Consistency in Neighborhood')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Cannot create visualizations: frame data or depth analysis not available")

## Detailed Neighborhood Analysis

In [ ]:
# Show examples of points with different depth situations
if frame_data is not None and depth_analysis is not None:
    # Find examples: point with valid depth, point with invalid depth, point with good neighborhood, etc.
    points_2d = visible_points['points_2d']
    point_valid = depth_analysis['point_valid']
    neighbor_valid_ratios = depth_analysis['neighbor_valid_ratios']
    
    # Find interesting examples
    examples = {}
    
    # Example 1: Point with valid depth
    valid_indices = np.where(point_valid)[0]
    if len(valid_indices) > 0:
        examples['valid_point'] = valid_indices[0]
    
    # Example 2: Point with invalid depth but good neighborhood
    invalid_indices = np.where(~point_valid)[0]
    if len(invalid_indices) > 0:
        # Find one with high neighbor valid ratio
        invalid_with_good_neighbors = invalid_indices[neighbor_valid_ratios[invalid_indices] > 0.5]
        if len(invalid_with_good_neighbors) > 0:
            examples['invalid_but_good_neighbors'] = invalid_with_good_neighbors[0]
        else:
            examples['invalid_point'] = invalid_indices[0]
    
    # Example 3: Point with poor neighborhood
    poor_neighbor_indices = np.where(neighbor_valid_ratios < 0.3)[0]
    if len(poor_neighbor_indices) > 0:
        examples['poor_neighborhood'] = poor_neighbor_indices[0]
    
    # Visualize examples
    if len(examples) > 0:
        fig, axes = plt.subplots(1, len(examples), figsize=(6*len(examples), 6))
        if len(examples) == 1:
            axes = [axes]
        
        for idx, (name, point_idx) in enumerate(examples.items()):
            ax = axes[idx]
            x, y = points_2d[point_idx]
            x_int, y_int = int(np.round(x)), int(np.round(y))
            
            # Extract neighborhood
            window_size = 5
            half = window_size // 2
            H, W = frame_data['depth'].shape
            x0 = max(0, x_int - half)
            x1 = min(W, x_int + half + 1)
            y0 = max(0, y_int - half)
            y1 = min(H, y_int + half + 1)
            
            # Get RGB and depth patches
            rgb_patch = frame_data['rgb'][y0:y1, x0:x1]
            depth_patch = frame_data['depth'][y0:y1, x0:x1]
            
            # Show RGB patch
            ax.imshow(rgb_patch)
            center_x, center_y = x_int - x0, y_int - y0
            ax.scatter([center_x], [center_y], c='red', s=200, marker='x', linewidths=3)
            ax.set_title(f'{name}\\n'
                        f'Point valid: {point_valid[point_idx]}\\n'
                        f'Neighbor valid ratio: {neighbor_valid_ratios[point_idx]:.2%}\\n'
                        f'Depth at point: {depth_analysis["point_depths"][point_idx]:.3f}m' if point_valid[point_idx] else 'Depth at point: N/A')
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\\nSummary Statistics:")
        print(f"  Total visible points: {len(points_2d)}")
        print(f"  Points with valid depth: {np.sum(point_valid)} ({100*np.mean(point_valid):.1f}%)")
        print(f"  Points with invalid depth: {np.sum(~point_valid)} ({100*np.mean(~point_valid):.1f}%)")
        print(f"  Mean neighbor valid ratio: {np.mean(neighbor_valid_ratios):.2%}")
        print(f"  Points with >50% valid neighbors: {np.sum(neighbor_valid_ratios > 0.5)} ({100*np.mean(neighbor_valid_ratios > 0.5):.1f}%)")
        print(f"  Points with <30% valid neighbors: {np.sum(neighbor_valid_ratios < 0.3)} ({100*np.mean(neighbor_valid_ratios < 0.3):.1f}%)")
        
        # Depth statistics for valid points
        valid_depths = depth_analysis['point_depths'][point_valid]
        if len(valid_depths) > 0:
            print(f"\\nDepth Statistics (valid points only):")
            print(f"  Mean depth: {np.mean(valid_depths):.3f}m")
            print(f"  Std depth: {np.std(valid_depths):.3f}m")
            print(f"  Min depth: {np.min(valid_depths):.3f}m")
            print(f"  Max depth: {np.max(valid_depths):.3f}m")
    else:
        print("No examples found to visualize")
else:
    print("Cannot create detailed analysis: frame data or depth analysis not available")